# 기본 확인

metadata 로드 - 전처리(시간,'Capacity','Re','Rct'수치형 변환) - 결측치 확인 - 시간 확인

데이터 나눔처리 ('B0005', 'B0006', 'B0007', 'B0018' 배터리 대상으로 충전 / 방전 / 임피던스별 data) 

In [1]:
# 도구 불러오기
import pandas as pd
import glob
import os
import re
import ast
from glob import glob
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib.font_manager as fm

# 판다스 출력 제한 해제 
pd.set_option('display.max_rows', 100) # 최대 100행까지 생략 없이 출력
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

In [6]:
import os
print(os.getcwd()) # 현재 실행 경로 출력

c:\python_course\3_11_project\sparta_project_3_team11\works\rsh


In [7]:
# metadata 로드 확인
df_meta = pd.read_csv("../../dataset/metadata.csv")
df_meta

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,[2010. 9. 30. 7. 36. ...,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,[2010. 9. 30. 8. 8. ...,4,B0055,248,7562,07562.csv,1.0201379996149256,NaN,NaN
7562,charge,[2010. 9. 30. 8. 48. 54.25],4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,[2010. 9. 30. 11. 50. ...,4,B0055,250,7564,07564.csv,0.9907591663373165,NaN,NaN


In [8]:
# 데이터 확인 (null/Dtype)
df_meta.info()

#-> 시간 object & Capacity,Re,Rct objct & Capacity,Re,Rct null 확인

<class 'pandas.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   type                 7565 non-null   str  
 1   start_time           7565 non-null   str  
 2   ambient_temperature  7565 non-null   int64
 3   battery_id           7565 non-null   str  
 4   test_id              7565 non-null   int64
 5   uid                  7565 non-null   int64
 6   filename             7565 non-null   str  
 7   Capacity             2794 non-null   str  
 8   Re                   1956 non-null   str  
 9   Rct                  1956 non-null   str  
dtypes: int64(3), str(7)
memory usage: 591.1 KB


In [9]:
# metadata 시간 전처리(1) 후 df로 저장

# 원본 CSV 불러오기
df_o = pd.read_csv("../../dataset/metadata.csv")

# start_time 전처리 함수
def parse_start_time(x):
    try:
        # 1. 대괄호 제거 및 공백 정리
        x_str = str(x).replace('[', '').replace(']', '').strip()
        
        # 2. 정규표현식으로 숫자(지수 표현 포함) 형태만 모두 추출
        # 예: '2.0080e+03', '4.0000e+00' 등을 찾아냄
        parts_raw = x_str.split()
        
        # 3. float로 먼저 변환해야 2.0080e+03 -> 2008.0 이 됩니다.
        # 그 후 int로 변환
        parts = [int(float(p)) for p in parts_raw]
        
        if len(parts) >= 5:
            # 초(parts[5])가 없을 수도 있으니 안전하게 시/분까지만 넣거나 조건부 추가
            return pd.Timestamp(year=parts[0], month=parts[1], day=parts[2],
                                hour=parts[3], minute=parts[4])
    except:
        return pd.NaT
    return pd.NaT

# start_time 컬럼 적용
df_meta['start_time'] = df_meta['start_time'].apply(parse_start_time)

# df로 저장
df = df_meta.copy()
display(df)

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,2010-07-21 15:00:00,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,2010-07-21 16:53:00,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,2010-07-21 17:25:00,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,2010-07-21 20:31:00,24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,2010-07-21 21:02:00,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,2010-09-30 07:36:00,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,2010-09-30 08:08:00,4,B0055,248,7562,07562.csv,1.0201379996149256,NaN,NaN
7562,charge,2010-09-30 08:48:00,4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,2010-09-30 11:50:00,4,B0055,250,7564,07564.csv,0.9907591663373165,NaN,NaN


In [10]:
# 전처리 (2. Dtype 변환)
# 'Capacity', 'Re', 'Rct' 컬럼을 수치형으로 변환
cols = ['Capacity', 'Re', 'Rct']

for col in cols:
    # errors='coerce'를 사용하면 숫자로 바꿀 수 없는 문자열을 NaN으로 처리합니다.
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 확인
print(df[cols].dtypes)

Capacity    float64
Re          float64
Rct         float64
dtype: object


In [11]:
# 변환 후 데이터 확인 (null/Dtype)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   type                 7565 non-null   str           
 1   start_time           7565 non-null   datetime64[us]
 2   ambient_temperature  7565 non-null   int64         
 3   battery_id           7565 non-null   str           
 4   test_id              7565 non-null   int64         
 5   uid                  7565 non-null   int64         
 6   filename             7565 non-null   str           
 7   Capacity             2769 non-null   float64       
 8   Re                   1947 non-null   float64       
 9   Rct                  1947 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(3), str(3)
memory usage: 591.1 KB


In [12]:
# 결측치 확인 (1.  discharge데이터에 Capacity의 결측치 여부 확인)
# 1. Discharge 데이터만 필터링
df_dis = df[df['type'] == 'discharge']

# 2. Capacity가 NaN인 행 추출
nan_capacity = df_dis[df_dis['Capacity'].isna()]

print(f"전체 Discharge 행 수: {len(df_dis):,}개")
print(f"Capacity가 NaN인 행 수: {len(nan_capacity):,}개")

if len(nan_capacity) > 0:
    print("결측치가 발견된 배터리 및 파일 목록:")
    # 어떤 파일에서 용량이 빠졌는지 요약 출력
    display(nan_capacity[['battery_id', 'filename']].drop_duplicates())
else:
    print("\n Discharge 데이터에 Capacity 값이 정상적으로 들어있습니다.")
# 확인시 25개의 결측치가 있었지만, 해당 배터리는 대상이 아님

전체 Discharge 행 수: 2,794개
Capacity가 NaN인 행 수: 25개
결측치가 발견된 배터리 및 파일 목록:


,battery_id,filename
4370,B0050,04371.csv
4372,B0050,04373.csv
4374,B0050,04375.csv
4376,B0050,04377.csv
4390,B0052,04391.csv
4394,B0052,04395.csv
4396,B0052,04397.csv
4398,B0052,04399.csv
4400,B0052,04401.csv
4402,B0052,04403.csv


In [13]:
# 결측치 확인 (2. impedance 데이터에 Re, RCT의 결측치 여부 확인)
# 1. Impedance 데이터만 필터링
df_imp = df[df['type'] == 'impedance']

# 2. Re 또는 Rct가 NaN인 행 추출
nan_imp = df_imp[df_imp['Re'].isna() | df_imp['Rct'].isna()]

print(f"전체 Impedance 행 수: {len(df_imp):,}개")
print(f"Re/Rct 결측 행 수: {len(nan_imp):,}개")

if len(nan_imp) > 0:
    print("\n결측치가 발견된 배터리 및 파일 목록 (상위 10개):")
    # 중복을 제거하고 파일 단위로 요약
    summary = nan_imp.groupby(['battery_id', 'filename']).size().reset_index(name='missing_rows')
    display(summary.head(10))
else:
    print("\n Impedance 데이터에 Re, Rct 값이 채워져 있습니다.")

# 3. (참고) 값의 범위 확인 (0이나 음수가 있는지 체크)
if len(df_imp) > len(nan_imp):
    print("\n--- [Re, Rct 값의 범위 요약] ---")
    display(df_imp[['Re', 'Rct']].describe())

# 결측이 있었으나, 해당 배터리는 대상이 아님

전체 Impedance 행 수: 1,956개
Re/Rct 결측 행 수: 9개

결측치가 발견된 배터리 및 파일 목록 (상위 10개):


,battery_id,filename,missing_rows
0,B0049,04268.csv,1
1,B0049,04280.csv,1
2,B0049,04282.csv,1
3,B0049,04292.csv,1
4,B0049,04294.csv,1
5,B0049,04306.csv,1
6,B0049,04316.csv,1
7,B0049,04318.csv,1
8,B0051,04454.csv,1



--- [Re, Rct 값의 범위 요약] ---


,Re,Rct
count,1.947000e+03,1.947000e+03
mean,-4.976500e+11,1.055903e+12
std,2.195872e+13,4.659154e+13
min,-9.689245e+14,-2.091081e+02
25%,5.782157e-02,8.155754e-02
50%,7.255344e-02,1.014191e-01
75%,9.229960e-02,1.565123e-01
max,4.482291e+02,2.055843e+15


In [14]:
# 데이터 시간 범위 확인
# 1. 전체 시간 범위 확인 (Min, Max)
start_date = df['start_time'].min()
end_date = df['start_time'].max()
duration = end_date - start_date

print("=== [데이터셋 시간 범위 결과] ===")
print(f"전체 시작 시점: {start_date}")
print(f"전체 종료 시점: {end_date}")
print(f"총 실험 기간  : {duration.days}일 {duration.seconds // 3600}시간")

# 2. 배터리별 시간 범위 확인 (각 배터리마다 실험 시점이 다를 수 있음)
print("\n=== [배터리 ID별 시간 범위] ===")
battery_time_range = df.groupby('battery_id')['start_time'].agg(['min', 'max', 'count', 'nunique'])
battery_time_range.columns = ['시작 시간', '종료 시간', '행 수', '파일 수']
display(battery_time_range)

# 전체 2008-04-02~2010-09-30 데이터였고,
# 해당 배터리는 2008-04-02~2008-05-28 (616사이클) & 07-07~08-20 (319사이클)이었음

=== [데이터셋 시간 범위 결과] ===
전체 시작 시점: 2008-04-02 13:08:00
전체 종료 시점: 2010-09-30 15:32:00
총 실험 기간  : 911일 2시간

=== [배터리 ID별 시간 범위] ===


,시작 시간,종료 시간,행 수,파일 수
battery_id,,,,
B0005,2008-04-02 13:08:00,2008-05-28 11:09:00,616,616
B0006,2008-04-02 13:08:00,2008-05-28 11:09:00,616,616
B0007,2008-04-02 13:08:00,2008-05-28 11:09:00,616,616
B0018,2008-07-07 12:26:00,2008-08-20 08:37:00,319,319
B0025,2009-02-13 19:03:00,2009-03-19 12:55:00,80,80
B0026,2009-02-13 19:03:00,2009-03-19 12:55:00,80,80
B0027,2009-02-13 19:03:00,2009-03-19 12:55:00,80,80
B0028,2009-02-13 19:03:00,2009-03-19 12:55:00,80,80
B0029,2009-04-07 15:59:00,2009-04-18 01:29:00,97,97


In [ ]:
# 'B0005', 'B0006', 'B0007', 'B0018' 배터리 대상으로 충전 / 방전 / 임피던스별 data 구성하기

# 대상 배터리 ID 리스트
target_battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']

# 데이터 저장할 폴더 경로
data_folder = "../../dataset/data"

# 데이터를 쌓아두기 위한 저장소(Dictionary) 생성
collected_data = {}

for battery_id in target_battery_ids:
    filtered_df = df[df['battery_id'] == battery_id]
    
    for _, row in filtered_df.iterrows():
        filename = row['filename']
        file_path = os.path.join(data_folder, filename)
        
        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path)

            # 정보 결합 (기존 로직 유지)
            temp_df['type'] = row['type']
            temp_df['start_time'] = row['start_time']
            temp_df['ambient_temperature'] = row['ambient_temperature']
            temp_df['battery_id'] = row['battery_id']
            temp_df['cycle_index'] = row.get('cycle_index', 0) # 아까 만든 사이클 번호가 있다면 활용

            if row['type'] == 'discharge':
                temp_df['Capacity'] = row['Capacity']
            elif row['type'] == 'impedance':
                temp_df['Re'] = row['Re']
                temp_df['Rct'] = row['Rct']

            # 데이터를 리스트에 추가
            key = f"df_{row['type']}_{battery_id}"
            if key not in collected_data:
                collected_data[key] = []
            collected_data[key].append(temp_df)

# 리스트에 모인 데이터들을 하나로 합치기
for key, df_list in collected_data.items():
    globals()[key] = pd.concat(df_list, ignore_index=True)
    print(f"✅ {key} 생성 완료: {globals()[key].shape} 행/열")

✅ df_charge_B0005 생성 완료: (541173, 11) 행/열
✅ df_discharge_B0005 생성 완료: (50285, 12) 행/열
✅ df_impedance_B0005 생성 완료: (13344, 12) 행/열
✅ df_charge_B0006 생성 완료: (541173, 11) 행/열
✅ df_discharge_B0006 생성 완료: (50285, 12) 행/열
✅ df_impedance_B0006 생성 완료: (13344, 12) 행/열
✅ df_charge_B0007 생성 완료: (541173, 11) 행/열
✅ df_discharge_B0007 생성 완료: (50285, 12) 행/열
✅ df_impedance_B0007 생성 완료: (13344, 12) 행/열
✅ df_charge_B0018 생성 완료: (279810, 11) 행/열
✅ df_impedance_B0018 생성 완료: (2544, 12) 행/열
✅ df_discharge_B0018 생성 완료: (34866, 12) 행/열


In [16]:
# 간단 확인
display(df_charge_B0006.head())
display(df_discharge_B0007.head())
display(df_impedance_B0005.head()) 

,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time,type,start_time,ambient_temperature,battery_id,cycle_index
0,3.864624,0.000082,24.682214,-0.001,-0.007,0.000,charge,2008-04-02 13:08:00,24,B0006,0
1,3.469113,-4.059185,24.695407,-4.060,1.558,2.532,charge,2008-04-02 13:08:00,24,B0006,0
2,3.994806,1.513750,24.711491,1.506,4.710,5.500,charge,2008-04-02 13:08:00,24,B0006,0
3,4.005888,1.511389,24.739672,1.506,4.726,8.344,charge,2008-04-02 13:08:00,24,B0006,0
4,4.012944,1.510817,24.753180,1.506,4.737,11.125,charge,2008-04-02 13:08:00,24,B0006,0


,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,type,start_time,ambient_temperature,battery_id,cycle_index,Capacity
0,4.199360,-0.001866,23.937044,-0.0004,0.000,0.000,discharge,2008-04-02 15:25:00,24,B0007,0,1.891052
1,4.199497,-0.002139,23.924074,-0.0004,4.215,16.781,discharge,2008-04-02 15:25:00,24,B0007,0,1.891052
2,3.985606,-1.988778,24.004257,-2.0000,3.003,35.703,discharge,2008-04-02 15:25:00,24,B0007,0,1.891052
3,3.963247,-1.992558,24.162868,-2.0000,2.987,53.781,discharge,2008-04-02 15:25:00,24,B0007,0,1.891052
4,3.946647,-1.988491,24.346368,-2.0000,2.972,71.922,discharge,2008-04-02 15:25:00,24,B0007,0,1.891052


,Sense_current,Battery_current,Current_ratio,Battery_impedance,Rectified_Impedance,type,start_time,ambient_temperature,battery_id,cycle_index,Re,Rct
0,(-1+1j),(-1+1j),(1+0j),(-0.43892624830326377-0.107298295835479j),(0.07006937798290404-0.00047998469078178944j),impedance,2008-04-18 20:55:00,24,B0005,0,0.044669,0.069456
1,(820.6094970703125-36.23455047607422j),(337.0914611816406-82.9207763671875j),(2.3204145178633437+0.4633045948164565j),(0.13008840651776496-0.19711481029612374j),(0.06817886114940203-0.001190040925296937j),impedance,2008-04-18 20:55:00,24,B0005,0,0.044669,0.069456
2,(827.2421875-48.23122787475586j),(330.6315612792969-70.01371765136719j),(2.424192647592199+0.36746495469515333j),(0.058770560504133235+0.03330656583655633j),(0.06793257733714593-5.6826811936507056e-05j),impedance,2008-04-18 20:55:00,24,B0005,0,0.044669,0.069456
3,(827.1934814453125-56.195716857910156j),(330.8086242675781-61.73442459106445j),(2.4470021712116985+0.28677775364826635j),(0.0058135116366746726-0.060546548141956195j),(0.06691839226387165-0.0008787264015490232j),impedance,2008-04-18 20:55:00,24,B0005,0,0.044669,0.069456
4,(824.9295043945312-53.241477966308594j),(332.68267822265625-57.62901306152344j),(2.434304977711638+0.2616460702282485j),(0.12608106668700975-0.09044390544679616j),(0.06807105294348659-0.0001974802021297548j),impedance,2008-04-18 20:55:00,24,B0005,0,0.044669,0.069456


In [17]:
# 분석가 팁: 반복되는 작업을 줄이기 위해 리스트를 순회하며 출력합니다.

# 1. 방전(Discharge) 데이터 확인
print("=== [DISCHARGE DATA SHAPE & PREVIEW] ===")
discharge_dfs = [df_discharge_B0005, df_discharge_B0006, df_discharge_B0007, df_discharge_B0018]
df_names_dis = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_dis, discharge_dfs):
    print(f"Discharge_{name} Shape: {df_obj.shape}")
    display(df_obj.head())  # 전체 출력 대신 head()를 권장하지만, 원하시면 그대로 사용 가능합니다.
    print("-" * 50)

# 2. 임피던스(Impedance) 데이터 확인
print("\n=== [IMPEDANCE DATA SHAPE & PREVIEW] ===")
impedance_dfs = [df_impedance_B0005, df_impedance_B0006, df_impedance_B0007, df_impedance_B0018]
df_names_imp = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_imp, impedance_dfs):
    print(f"Impedance_{name} Shape: {df_obj.shape}")
    display(df_obj.head())
    print("-" * 50)

# 3. 충전(Charge) 데이터 확인
print("\n=== [CHARGE DATA SHAPE & PREVIEW] ===")
charge_dfs = [df_charge_B0005, df_charge_B0006, df_charge_B0007, df_charge_B0018]
df_names_cha = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_cha, charge_dfs):
    print(f"Charge_{name} Shape: {df_obj.shape}")
    display(df_obj.head())
    print("-" * 50)

=== [DISCHARGE DATA SHAPE & PREVIEW] ===
Discharge_B0005 Shape: (50285, 12)


,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,type,start_time,ambient_temperature,battery_id,cycle_index,Capacity
0,4.191492,-0.004902,24.330034,-0.0006,0.000,0.000,discharge,2008-04-02 15:25:00,24,B0005,0,1.856487
1,4.190749,-0.001478,24.325993,-0.0006,4.206,16.781,discharge,2008-04-02 15:25:00,24,B0005,0,1.856487
2,3.974871,-2.012528,24.389085,-1.9982,3.062,35.703,discharge,2008-04-02 15:25:00,24,B0005,0,1.856487
3,3.951717,-2.013979,24.544752,-1.9982,3.030,53.781,discharge,2008-04-02 15:25:00,24,B0005,0,1.856487
4,3.934352,-2.011144,24.731385,-1.9982,3.011,71.922,discharge,2008-04-02 15:25:00,24,B0005,0,1.856487


--------------------------------------------------
Discharge_B0006 Shape: (50285, 12)


,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,type,start_time,ambient_temperature,battery_id,cycle_index,Capacity
0,4.179800,-0.002366,24.277568,-0.0006,0.000,0.000,discharge,2008-04-02 15:25:00,24,B0006,0,2.035338
1,4.179823,0.000434,24.277073,-0.0006,4.195,16.781,discharge,2008-04-02 15:25:00,24,B0006,0,2.035338
2,3.966528,-2.014242,24.366226,-1.9990,3.070,35.703,discharge,2008-04-02 15:25:00,24,B0006,0,2.035338
3,3.945886,-2.008730,24.515123,-1.9990,3.045,53.781,discharge,2008-04-02 15:25:00,24,B0006,0,2.035338
4,3.930354,-2.013381,24.676053,-1.9990,3.026,71.922,discharge,2008-04-02 15:25:00,24,B0006,0,2.035338


--------------------------------------------------
Discharge_B0007 Shape: (50285, 12)


...

--------------------------------------------------
Charge_B0007 Shape: (541173, 11)


,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time,type,start_time,ambient_temperature,battery_id,cycle_index
0,3.866123,-0.003830,24.434244,-0.0006,0.002,0.000,charge,2008-04-02 13:08:00,24,B0007,0
1,3.644900,-2.261867,24.441053,-2.2697,2.576,2.532,charge,2008-04-02 13:08:00,24,B0007,0
2,4.001099,1.489161,24.445727,1.4995,4.719,5.500,charge,2008-04-02 13:08:00,24,B0007,0
3,4.011041,1.491029,24.459603,1.4995,4.745,8.344,charge,2008-04-02 13:08:00,24,B0007,0
4,4.017485,1.491413,24.458385,1.4995,4.745,11.125,charge,2008-04-02 13:08:00,24,B0007,0


--------------------------------------------------
Charge_B0018 Shape: (279810, 11)


,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time,type,start_time,ambient_temperature,battery_id,cycle_index
0,3.865713,0.001014,23.735721,0.000,-0.007,0.000,charge,2008-07-07 12:26:00,24,B0018,0
1,3.447651,-4.034445,23.743956,-4.036,1.553,2.484,charge,2008-07-07 12:26:00,24,B0018,0
2,4.005559,1.517435,23.773723,1.507,4.721,5.109,charge,2008-07-07 12:26:00,24,B0018,0
3,4.015989,1.514558,23.777077,1.507,4.737,7.562,charge,2008-07-07 12:26:00,24,B0018,0
4,4.023230,1.517284,23.792710,1.507,4.743,10.062,charge,2008-07-07 12:26:00,24,B0018,0


--------------------------------------------------


# 분석